In [1]:
import torch
import numpy as np
from cryo_sbi.utils.infer_populations import center_models

In [26]:
models_all = torch.load("hsp90_models.pt")

# Select only 4 models
selected_indices = [0, 6, 12, 18]  # example for 20 original states

# Extract only these 4 models
models = models_all[selected_indices, :, :]  # shape [4, 3, 1207]

n_states = len(models)


In [27]:
torch.save(models, "hsp90_models_4.pt")

In [2]:
max_model = torch.load('hsp90_models-small.pt')

In [3]:
# center models (just to be sure)
models = center_models(max_model)
torch.save(models, "models.pt")
#torch.save(models, "RESNET_18_FFT_Filter/models.pt")
# torch.save(models, "RESNET_18/embed_dim_16/models.pt")
# torch.save(models, "RESNET_18/embed_dim_64/models.pt")
# torch.save(models, "RESNET_18/embed_dim_256/models.pt")

#torch.save(models, "SPATIAL_CRYO/models.pt")
#torch.save(models, "SPATIAL_CRYO_FFT_Filter/models.pt")
#torch.save(models, "SPATIAL_CRYO_GAUSS_FFT_FILTER/models.pt")

In [2]:
def generate_population_weights(n_states, population_steps):
    """
    Generate population weights for mixtures of state 0 and each other state.
    Returns:
        weights: (n_points, n_states)
        pop_fractions: fraction of state 0 for each mixture
        state_pairs: list of tuples (0, i) for each mixture
    """
    fractions = np.linspace(0, 1, population_steps)
    weights_list = []
    pop_fractions = []
    state_pairs = []

    for i in range(1, n_states):
        for f in fractions:
            w = np.zeros(n_states)
            w[0] = f
            w[i] = 1 - f
            weights_list.append(w)
            pop_fractions.append(f)
            state_pairs.append((0, i))

    weights = np.array(weights_list)
    pop_fractions = np.array(pop_fractions)
    state_pairs = np.array(state_pairs)

    weights, idx = np.unique(weights, axis=0, return_index=True)
    pop_fractions = pop_fractions[idx]
    state_pairs = state_pairs[idx]

    return weights, pop_fractions, state_pairs